In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt

from dotenv import load_dotenv
import os

from dataset_classes.lazy_dataset import LazyDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

In [ ]:
load_dotenv("config.env")

# --- Dataset parameters ---
USE_DVS = int(os.getenv("USE_DVS"))
USE_APS = int(os.getenv("USE_APS"))

HEIGHT = int(os.getenv("HEIGHT"))
WIDTH = int(os.getenv("WIDTH"))
APS_FRAME_NAME = f"aps_frame_{HEIGHT}x{WIDTH}"
DVS_FRAME_NAME = f"dvs_frame_{HEIGHT}x{WIDTH}"

PRECISION = int(os.getenv("PRECISION"))
DTYPE = np.float16 if PRECISION == 16 else np.float32

# --- Location parameters ---
DATA_DIR = os.getenv("DATA_DIR")
TRAINING_DIR = f"{DATA_DIR}/Training"
TEST_DIR = f"{DATA_DIR}/Test"

BATCH_SIZE = int(os.getenv("BATCH_SIZE"))
WINDOW_SIZE = int(os.getenv("WINDOW_SIZE"))

NORMALISE_OUTPUT = False

In [ ]:
inv_apsdvs_max = 1 / 255

# These normalisation functions are not standardised intentionally, to take advantage of pre-computing with some known values.
# used to normalise the aps and dvs frames to the range [0, 1]
def normalise_dvsaps(data):
    # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
    new_data = np.empty_like(data, dtype=DTYPE)
    new_data = data * inv_apsdvs_max
    return new_data

transform = transforms.Compose([
    lambda x: normalise_dvsaps(x),
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

# normalise the steering values from -180 to 180 to -1 to 1
# Precompute normalization coefficients
steering_slope = 1 / 180  # (output_max - output_min)/(input_max - input_min) = (1-(-1))/(180-(-180))
def normalise_steering(data):
    return data * steering_slope

# used for getting the data out at the end.
inv_steering_slope = 180
def denormalise_steering(data):
    return data * inv_steering_slope

if NORMALISE_OUTPUT:
    target_transform = transforms.Compose([
        lambda x: normalise_steering(x)
    ])
else:
   target_transform = transforms.Compose([]) 

train_dataset = LazyDataset(TRAINING_DIR, transform=transform, target_transform=target_transform, window_size=1, use_aps=USE_APS, use_dvs=USE_DVS, DVS_FRAME_NAME=DVS_FRAME_NAME, APS_FRAME_NAME=APS_FRAME_NAME, DTYPE=DTYPE)
test_dataset = LazyDataset(TEST_DIR, transform=transform, target_transform=target_transform, window_size=1, use_aps=USE_APS, use_dvs=USE_DVS, DVS_FRAME_NAME=DVS_FRAME_NAME, APS_FRAME_NAME=APS_FRAME_NAME, DTYPE=DTYPE)

# load the data, can implement batching etc. here
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=4, pin_memory=True, prefetch_factor=5)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=4, pin_memory=True, prefetch_factor=5)

In [ ]:
fig = plt.figure(figsize=(20, 12))
steering_angles = []

for combined_images, steering_angle in train_loader:
    steering_angle = steering_angle.squeeze(1)
    results = steering_angle.numpy()
    if NORMALISE_OUTPUT:
        results = denormalise_steering(results)
    steering_angles.extend(results)

# Plotting the steering angles
# plt.plot(range(len(steering_angles)), steering_angles)
plt.hist(steering_angles, bins=360, alpha=0.7)
plt.title('Distribution of Steering Angles')
plt.xlabel('Steering Angle')
plt.ylabel('Frequency')
plt.show()